Cell 1: Settings: Drive paths, repo and branch

In [ ]:
# Google Drive locations
DRIVE_ROOT = "/content/drive/MyDrive/Documents/ACIT4030_Group_Assignment 2"   # results end up here
DATA_ZIP   = DRIVE_ROOT + "/data/acit4030-data.zip"                     # The dataset zip. Same structure as the one from Canvas
PACKAGES   = "/content/drive/MyDrive/Colab_Packages"                    # PyTorch3D lives here

# GitHub
REPO_URL      = "github.com/SebastianHaugen/acit4030-NeRF.git"   # repo
BRANCH        = "jesse-branch"                           # the branch to train from
TOKEN_SECRET  = "token_github"                           # name of the secret in the Colab Secrets panel

REPO_DIR = "/content/acit4030-NeRF"

Cell 2: Check the runtime and GPU

In [ ]:
import sys, torch

print(f"Python  {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}  (CUDA {torch.version.cuda})")
assert torch.cuda.is_available(), "No GPU - switch the runtime type to GPU"
print(f"GPU     {torch.cuda.get_device_name(0)}")

Cell 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Cell 4: Load PyTorch3D from Drive and check compatibility

In [ ]:
import os
from pathlib import Path

os.makedirs(PACKAGES, exist_ok=True)
for p in (PACKAGES, PACKAGES + "/pytorch3d"):
    if p not in sys.path:
        sys.path.append(p)
os.environ["PYTHONPATH"] = PACKAGES + ":" + PACKAGES + "/pytorch3d"   # for !python subprocesses

runtime = f"Python {sys.version.split()[0]} | torch {torch.__version__} | CUDA {torch.version.cuda}"
build_file = Path(PACKAGES, "build_info.txt")
built_for = build_file.read_text().strip() if build_file.exists() else "unknown"
print("Runtime now: ", runtime)
print("Built for:   ", built_for)

try:
    from pytorch3d import _C   # the compiled part, fails if versions don't match
    import pytorch3d
    print(f"\nPyTorch3D {pytorch3d.__version__} works.")
except Exception as e:
    print(f"\nPyTorch3D not usable ({type(e).__name__}). Set INSTALL_PYTORCH3D = True below and run cell 5 once.")

Cell 5: Install PyTorch3D to Drive (only if cell 4 fails)

In [ ]:
INSTALL_PYTORCH3D = False

if INSTALL_PYTORCH3D:
    !pip install 'git+https://github.com/facebookresearch/pytorch3d.git@stable' --target="{PACKAGES}" --upgrade
    build_file.write_text(runtime)
    print("Installed for", runtime, "- now restart the session.")
else:
    print("Skipped.")

Cell 6: Clone or update the repository

In [ ]:
from google.colab import userdata

token = userdata.get(TOKEN_SECRET)
if not os.path.exists(REPO_DIR):
    !git clone -q https://{token}@{REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin
!git checkout -q {BRANCH}
!git pull -q origin {BRANCH}
!git log -1 --format="On %D: %h %s"

Cell 7: Install Python packages from requirements.txt

In [ ]:
!pip install -q -r requirements.txt

Cell 8: Unzip the dataset into data/

In [ ]:
import shutil, subprocess

data_dir = Path(REPO_DIR, "data")

def dataset_ready():
    return (data_dir / "lego" / "transforms_train.json").exists() and \
           (data_dir / "poster" / "transforms.json").exists()

if not dataset_ready():
    tmp = Path("/content/_unzipped")
    shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run(["unzip", "-q", DATA_ZIP, "-d", str(tmp)], check=True)
    data_dir.mkdir(exist_ok=True)
    for name in ("lego", "poster"):
        found = [p for p in tmp.rglob(name) if p.is_dir() and "__MACOSX" not in p.parts]
        assert found, f"No '{name}' folder in the zip"
        shutil.rmtree(data_dir / name, ignore_errors=True)
        shutil.move(str(found[0]), data_dir / name)
    shutil.rmtree(tmp)

print("Dataset ready:", dataset_ready())
for name in ("lego", "poster"):
    print(f"  {name}: {sorted(p.name for p in (data_dir / name).iterdir())}")

Cell 9: Link outputs/ to Google Drive

In [ ]:
for name in ("lego", "poster"):
    target = Path(DRIVE_ROOT, "outputs", name)
    target.mkdir(parents=True, exist_ok=True)
    link = Path(REPO_DIR, "outputs", name)
    if link.is_symlink():
        continue
    if link.exists():
        for item in link.iterdir():
            shutil.move(str(item), target / item.name)
        link.rmdir()
    link.symlink_to(target)

!ls -l outputs

Cell 10: Define helpers: show(), show_last_preview()

In [ ]:
from IPython.display import Image, display

def show(path, width=900):
    path = Path(REPO_DIR, path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print("Not found:", path)

def show_last_preview(run_dir, width=900):
    previews = sorted(Path(REPO_DIR, run_dir, "previews").glob("iter_*.png"))
    show(previews[-1], width) if previews else print("No previews in", run_dir)

!python -m src.repo_util.LoadConfigurations

Cell 11: Smoke test on lego (short run, checks that the loss goes down)

In [ ]:
!NERF_DATASET=lego python -m src.smoke_test
show_last_preview("outputs/smoke_test/lego")

Cell 12: Train on lego (iterations from config.yaml)

In [ ]:
!NERF_DATASET=lego python -m src.train
show_last_preview("outputs/lego")

Cell 13: Evaluate lego on test views

In [ ]:
!NERF_DATASET=lego python -m src.evaluate
show("report/figures/qualitative_lego.png")

Cell 14: Smoke test on poster (short run, checks that the loss goes down)

In [ ]:
!NERF_DATASET=poster python -m src.smoke_test
show_last_preview("outputs/smoke_test/poster", width=600)

Cell 15: Train on poster (iterations from config.yaml)

In [ ]:
!NERF_DATASET=poster python -m src.train
show_last_preview("outputs/poster", width=600)

Cell 16: Evaluate poster on test views

In [ ]:
!NERF_DATASET=poster python -m src.evaluate
show("report/figures/qualitative_poster.png", width=450)

Cell 17: Build report figures and table, copy to Drive

In [ ]:
!python -m src.report_figures
!cat report/figures/table_metrics.tex

figures_backup = Path(DRIVE_ROOT, "report_figures")
shutil.copytree(Path(REPO_DIR, "report", "figures"), figures_backup, dirs_exist_ok=True)
print("\nCopied to", figures_backup)
!ls "{figures_backup}"

Cell 18: Check that no tracked file was changed (config.yaml stays as committed)

In [ ]:
!git status --short